[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinod-seth/Applied-Scientist-Interview-Gauntlet/blob/main/tutorial/01_finetuning_and_architecture/02_architecture_unit_tests.ipynb)

# Armory Notebook 2 — Architecture Claims as Failing/Passing Tests

| | |
|---|---|
| **Companion to** | Lesson 2 — Defending the RoPE Transformer |
| **Runtime** | CPU is fine (pure PyTorch, tiny tensors) |
| **Estimated time** | 30 minutes |
| **XP** | **0.** Evidence, not defense. |
| **Last verified** | 2026-07-13 |

Every claim you make about your from-scratch transformer is a testable property. This notebook turns four of them into assertions: RoPE's shift invariance, why values must not be rotated, padding-mask equivalence, and the last-token indexing bug from Lesson 1's debugging question. Each test here is one you should port to *your own* model code — a failing port is a floor-hit found for free.

> [!NOTE]
> The implementation below is deliberately minimal and self-contained so the properties are legible. Where your implementation differs (dimension pairing convention, mask shape), adapt the test, not the claim.

In [ ]:
%pip install -q "torch>=2.4"   # Colab ships torch preinstalled; this is a no-op there

In [ ]:
import torch

torch.manual_seed(7)

def rotary_phases(positions: torch.Tensor, head_dim: int) -> tuple[torch.Tensor, torch.Tensor]:
    """cos/sin tables: one frequency per 2D plane, theta_i = 10000^(-2i/d).

    Phases are computed in float64 and cast down. In float32, an angle of
    ~5000 rad carries absolute error ~ angle * eps ~ 6e-4 - enough to visibly
    break shift invariance at large positions. Test 1's shift=5000 case fails
    without this; real RoPE implementations have exactly this trap.
    """
    plane_idx = torch.arange(head_dim // 2, dtype=torch.float64)
    freqs = 10000.0 ** (-2.0 * plane_idx / head_dim)
    angles = positions.double().unsqueeze(-1) * freqs       # [seq, d/2]
    return angles.cos().float(), angles.sin().float()

def rotate_pairs(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """Apply the per-plane 2D rotation. x: [seq, d]; planes are (even, odd) index pairs."""
    x_even, x_odd = x[..., 0::2], x[..., 1::2]
    return torch.stack(
        (x_even * cos - x_odd * sin, x_even * sin + x_odd * cos), dim=-1
    ).flatten(-2)

def attention_logits(q: torch.Tensor, k: torch.Tensor, positions: torch.Tensor) -> torch.Tensor:
    cos, sin = rotary_phases(positions, q.shape[-1])
    q_rot, k_rot = rotate_pairs(q, cos, sin), rotate_pairs(k, cos, sin)
    return (q_rot @ k_rot.T) / (q.shape[-1] ** 0.5)

SEQ, HEAD_DIM = 12, 64
q_proj_out = torch.randn(SEQ, HEAD_DIM)
k_proj_out = torch.randn(SEQ, HEAD_DIM)
print("Toy tensors ready:", q_proj_out.shape)

## Test 1 — Shift invariance: the property RoPE exists to provide

Lesson 2, Follow-up 2: logits between positions (m, n) must equal logits between (m+s, n+s). If this assertion fails on your model, your rotation, pairing, or position ids are wrong.

> [!NOTE]
> Building this notebook surfaced a genuine trap: with phases computed in float32, the shift=5000 case fails at atol=1e-4 — not because the math is wrong, but because cos/sin of large fp32 arguments lose ~`angle × eps` of absolute precision. The implementation above computes phases in float64 and casts down. If an interviewer asks "what numerical issues did you hit?", this is a first-class answer.

In [ ]:
base_positions = torch.arange(SEQ)
for shift in (1, 100, 5000):
    unshifted = attention_logits(q_proj_out, k_proj_out, base_positions)
    shifted   = attention_logits(q_proj_out, k_proj_out, base_positions + shift)
    max_gap = (unshifted - shifted).abs().max().item()
    assert torch.allclose(unshifted, shifted, atol=1e-4), f"shift={shift} broke invariance"
    print(f"shift={shift:>5}: PASS  (max |delta logit| = {max_gap:.2e})")
print("\nRelative position survives arbitrary absolute offsets - the derivation, executed.")

## Test 2 — Why values are never rotated

The counterfactual: rotate V too, and the *content* the attention sum reads now depends on absolute position. Shift invariance of the block output dies.

In [ ]:
v_proj_out = torch.randn(SEQ, HEAD_DIM)

def block_output(positions: torch.Tensor, rotate_values: bool) -> torch.Tensor:
    weights = attention_logits(q_proj_out, k_proj_out, positions).softmax(dim=-1)
    if rotate_values:
        cos, sin = rotary_phases(positions, HEAD_DIM)
        return weights @ rotate_pairs(v_proj_out, cos, sin)
    return weights @ v_proj_out

for rotate_v in (False, True):
    out_a = block_output(base_positions, rotate_v)
    out_b = block_output(base_positions + 100, rotate_v)
    gap = (out_a - out_b).abs().max().item()
    verdict = "invariant (correct)" if gap < 1e-4 else "POSITION-DEPENDENT (broken)"
    print(f"rotate V = {str(rotate_v):<5} -> max output delta under shift: {gap:.2e}  {verdict}")

## Test 3 — Padding-mask equivalence

A padded and an unpadded copy of the same sequence must produce identical outputs at the real positions. Loss can happily decrease with a broken mask — this five-liner is the only cheap way to know.

In [ ]:
def masked_attention(q, k, v, key_is_real: torch.Tensor) -> torch.Tensor:
    logits = (q @ k.T) / (q.shape[-1] ** 0.5)
    logits = logits.masked_fill(~key_is_real.unsqueeze(0), float("-inf"))
    return logits.softmax(dim=-1) @ v

REAL_LEN, PAD = 8, 4
pad_noise = torch.randn(PAD, HEAD_DIM)  # garbage the mask must neutralize
q_pad = torch.cat([q_proj_out[:REAL_LEN], pad_noise])
k_pad = torch.cat([k_proj_out[:REAL_LEN], pad_noise])
v_pad = torch.cat([v_proj_out[:REAL_LEN], pad_noise])
real_flags = torch.tensor([True] * REAL_LEN + [False] * PAD)

with_pad    = masked_attention(q_pad, k_pad, v_pad, real_flags)[:REAL_LEN]
without_pad = masked_attention(q_proj_out[:REAL_LEN], k_proj_out[:REAL_LEN],
                               v_proj_out[:REAL_LEN], torch.ones(REAL_LEN, dtype=torch.bool))
assert torch.allclose(with_pad, without_pad, atol=1e-5), "mask leaks padding into real positions"
print("PASS: padded and unpadded runs agree at every real position.")
print("Port this to your model: same sequence, two paddings, assert equality.")

## Test 4 — The last-token indexing bug, reproduced

Lesson 1's debugging question made this a thought experiment; here it is in tensors. Right-padded batch, classification head reading `hidden[:, -1]` versus the last *real* token.

In [ ]:
BATCH, MAX_LEN, HIDDEN, N_CLASSES = 4, 10, 32, 4
lengths = torch.tensor([10, 7, 4, 2])          # only sequence 0 fills the batch width
hidden_states = torch.randn(BATCH, MAX_LEN, HIDDEN)
pad_vector = torch.randn(HIDDEN)               # models emit near-constant states at pads
for row, true_len in enumerate(lengths):
    hidden_states[row, true_len:] = pad_vector + 0.01 * torch.randn(MAX_LEN - true_len, HIDDEN)

cls_head = torch.nn.Linear(HIDDEN, N_CLASSES)

naive_logits   = cls_head(hidden_states[:, -1])                                   # the bug
gather_idx     = (lengths - 1).view(-1, 1, 1).expand(-1, 1, HIDDEN)
correct_logits = cls_head(hidden_states.gather(1, gather_idx).squeeze(1))          # the fix

print("row  len  naive pred  correct pred")
for row in range(BATCH):
    print(f" {row}   {lengths[row].item():>3}      {naive_logits[row].argmax().item()}           "
          f"{correct_logits[row].argmax().item()}")
print("\nEvery short sequence collapses onto the pad vector's prediction under the naive index -")
print("the 'all outputs are the majority class at eval' mechanism, live.")

## Test 5 — The 1/√d_k factor as a variance measurement

Derivation says Var(q·k) = d_k for unit-variance components. Measure it, then watch what saturation does to softmax entropy.

In [ ]:
print(" d_k   logit std (raw)   logit std (scaled)   softmax entropy raw/scaled (nats)")
for d_k in (16, 64, 256, 1024):
    qs, ks = torch.randn(4000, d_k), torch.randn(4000, d_k)
    raw = (qs * ks).sum(-1)
    scaled = raw / d_k ** 0.5
    def entropy(logit_rows: torch.Tensor) -> float:
        probs = logit_rows.view(200, 20).softmax(-1)
        return -(probs * probs.clamp_min(1e-12).log()).sum(-1).mean().item()
    print(f"{d_k:>4}   {raw.std():>13.2f}   {scaled.std():>17.2f}   "
          f"{entropy(raw):.3f} / {entropy(scaled):.3f}")
print("\nRaw std tracks sqrt(d_k); scaled stays ~1. Raw softmax entropy collapses toward 0")
print("(one-hot, dead gradients) while scaled stays in the responsive regime.")

## Close the loop

1. Port Tests 1, 3, and 4 to your actual model repository. Every port that fails is a floor-hit — log it in [PROGRESS.md](../../PROGRESS.md) with the exact mechanism.
2. In the boss fight, "how do you *know* your mask is right?" now has a one-sentence answer: name the equivalence test and its tolerance.
3. Nothing here fills a Metric Vault row — these are correctness properties, not performance numbers. Their value is that they make your *qualitative* claims defensible to the floor.